# 作业 03：在 PyTorch 中调用交叉熵

本次只判断一件事：**有没有行人？** 不使用图片、CNN 或数据下载。我们用一个简单函数产生两个预测概率，把注意力放在最后的损失函数调用上。

类别的顺序始终固定：

| 类别编号 | 含义 | one-hot 标签 y |
|---|---|---|
| 0 | 没有行人 | `[1, 0]` |
| 1 | 有行人 | `[0, 1]` |

`y` 和 `y_pred` 都表示**同一个样本的两个类别**，shape 为 `(2,)`。`y_pred[0]` 是没有行人的概率，`y_pred[1]` 是有行人的概率，两者相加为 1。

目标：会调用 `F.cross_entropy`，并能解释概率变大或变小时，损失为什么变化。


In [1]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)

def show(name, value):
    print(f'{name}: {value}, shape={tuple(value.shape)}, dtype={value.dtype}')


## 1. 一个输出两个概率的预测函数

为了便于观察，直接把输入 x 当作“有行人的预测概率”，于是“没有行人的概率”就是 `1-x`。这是人为设置的教学函数，不是训练好的行人识别器。

先取 x=0.2，则预测应为 `[0.8, 0.2]`。


In [2]:
def predict(x):
    # x 是一个 0 到 1 之间的浮点标量张量
    return torch.stack([1 - x, x])

x = torch.tensor(0.2)
y = torch.tensor([1., 0.])  # 真实情况：没有行人
y_pred = predict(x)

show('x', x)
show('y（one-hot 真实标签）', y)
show('y_pred（两个预测概率）', y_pred)
show('预测概率之和', y_pred.sum())


x: 0.20000000298023224, shape=(), dtype=torch.float32
y（one-hot 真实标签）: tensor([1., 0.]), shape=(2,), dtype=torch.float32
y_pred（两个预测概率）: tensor([0.8000, 0.2000]), shape=(2,), dtype=torch.float32
预测概率之和: 1.0, shape=(), dtype=torch.float32


## 2. 最后调用 cross_entropy 计算损失

**PyTorch 的 `F.cross_entropy` 接收原始分数 logits，而不是直接接收概率。**

本例的函数已经输出概率，因此先用 `torch.log(y_pred)` 构造一组等价分数：对这组分数做 softmax，会还原原来的概率。这里两个概率均大于 0。

真实标签使用类别编号：`y=[1,0]` 对应编号 `0`，`y=[0,1]` 对应编号 `1`。用 `y.argmax()` 取出这个编号，得到整数标量。这里取的是**真实标签**的编号，不是预测的 argmax。

请重点看最后一行：损失由 `F.cross_entropy(...)` 计算。


In [3]:
scores = torch.log(y_pred)  # 从本例的概率构造等价 logits
label = y.argmax()        # [1, 0] → 类别编号 0，dtype 为 torch.int64

show('scores（传给损失函数的分数）', scores)
show('softmax(scores)（还原概率）', scores.softmax(dim=0))
show('label（真实类别编号）', label)

loss = F.cross_entropy(scores, label)
show('loss', loss)


scores（传给损失函数的分数）: tensor([-0.2231, -1.6094]), shape=(2,), dtype=torch.float32
softmax(scores)（还原概率）: tensor([0.8000, 0.2000]), shape=(2,), dtype=torch.float32
label（真实类别编号）: 0, shape=(), dtype=torch.int64
loss: 0.2231435477733612, shape=(), dtype=torch.float32


### 对照手算

展开公式：

$$L=-[y_0\ln(p_0)+y_1\ln(p_1)].$$

真实标签为 `[1,0]` 时，只留下 $-\ln(p_0)$。当前 $p_0=0.8$，所以损失约为 **0.2231**。

下面仅用于核对；实际调用损失函数时，仍使用上面的 `F.cross_entropy`。


In [4]:
manual_loss = -torch.log(y_pred[label])
show('手算：真实类别概率的负对数', manual_loss)
print('与 cross_entropy 一致：', torch.allclose(loss, manual_loss))


手算：真实类别概率的负对数: 0.2231435328722, shape=(), dtype=torch.float32
与 cross_entropy 一致： True


## 3. 几个带数字的例子

下面固定真实标签为 `[1,0]`，即“没有行人”。只改变模型的预测概率。

| 真实标签 y | 预测概率 y_pred | 交叉熵 |
|---|---|---|
| `[1,0]` | `[0.9,0.1]` | $-\ln(0.9)\approx0.1054$ |
| `[1,0]` | `[0.5,0.5]` | $-\ln(0.5)\approx0.6931$ |
| `[1,0]` | `[0.1,0.9]` | $-\ln(0.1)\approx2.3026$ |
| `[1,0]` | `[0,1]` | $-\ln(0)\to+\infty$：把真实类别的概率设为 0 |
| `[1,0]` | `[1,0]` | $-\ln(1)=0$：完全相信真实类别 |

概率越接近真实标签，损失越小。概率精确为 0 的情况是边界示例；PyTorch 会在下面的错误预测例子中打印 `inf`，表示正无穷。


In [5]:
def compare(y, probabilities):
    label = y.argmax()
    # 仅为展示指定概率的交叉熵。log(0)=-inf，是这里刻意保留的边界情况。
    scores = torch.log(probabilities)
    loss = F.cross_entropy(scores, label)
    print(f'y={y.tolist()}, y_pred={probabilities.tolist()}')
    print(f'真实类别概率={probabilities[label].item():.4f}, loss={loss.item():.4f}')
    return loss

y = torch.tensor([1., 0.])
example_losses = []
for values in [[0.9, 0.1], [0.5, 0.5], [0.1, 0.9], [0., 1.], [1., 0.]]:
    example_losses.append(compare(y, torch.tensor(values)))
    print()


y=[1.0, 0.0], y_pred=[0.8999999761581421, 0.10000000149011612]
真实类别概率=0.9000, loss=0.1054

y=[1.0, 0.0], y_pred=[0.5, 0.5]
真实类别概率=0.5000, loss=0.6931

y=[1.0, 0.0], y_pred=[0.10000000149011612, 0.8999999761581421]
真实类别概率=0.1000, loss=2.3026

y=[1.0, 0.0], y_pred=[0.0, 1.0]
真实类别概率=0.0000, loss=inf

y=[1.0, 0.0], y_pred=[1.0, 0.0]
真实类别概率=1.0000, loss=0.0000



## 4. 换成“有行人”试试看

现在 y 为 `[0,1]`，真实类别是 1，应当关注第二个概率。下面预测为 `[0.1,0.9]`，损失约为 **0.1054**。同一个预测在上节被惩罚，在这里得到较小损失，是因为真实答案变了。


In [6]:
y = torch.tensor([0., 1.])
y_pred = torch.tensor([0.1, 0.9])
compare(y, y_pred)


y=[0.0, 1.0], y_pred=[0.10000000149011612, 0.8999999761581421]
真实类别概率=0.9000, loss=0.1054


tensor(0.1054)

## 5. 真正的网络输出原始分数时，直接调用

前面的 `torch.log` 是为了配合本题“给定概率”的数值演示。真实网络通常直接输出两个原始分数，此时**直接把分数交给 cross_entropy**，不需要先 softmax 或取 log。

下面的 scores 为 `[2,-1]`，它们不是概率，也不需要加起来等于 1。softmax 只用来把概率打印给我们看，不是交叉熵的输入。


In [7]:
scores = torch.tensor([2., -1.])
y = torch.tensor([1., 0.])
label = y.argmax()

show('原始分数 scores', scores)
show('给人看的预测概率', scores.softmax(dim=0))
loss = F.cross_entropy(scores, label)
show('直接调用 cross_entropy 得到的 loss', loss)


原始分数 scores: tensor([ 2., -1.]), shape=(2,), dtype=torch.float32
给人看的预测概率: tensor([0.9526, 0.0474]), shape=(2,), dtype=torch.float32
直接调用 cross_entropy 得到的 loss: 0.04858732968568802, shape=(), dtype=torch.float32


## 你的实验（50分）

1. 把第 1 节的 x 改成 `0.8`，从第 1 节重新运行到手算核对。保持真实标签为 `[1,0]`。记录 y_pred 和损失，解释为什么损失变大。
2. 在下面补全代码：真实标签是“有行人”，预测概率为 `[0.8,0.2]`。打印两个张量的 shape，再调用 cross_entropy，预期损失约为 `1.6094`。
3. 用自己的话解释：为什么 `[1,0]` 对 `[0,1]` 的损失是无穷大，而 `[1,0]` 对 `[1,0]` 的损失是 0？
4. `F.cross_entropy(y_pred, label)` 与 `F.cross_entropy(torch.log(y_pred), label)` 中，哪一个才是在计算这里指定的预测概率对应的交叉熵？原始网络输出 scores 时，又该怎样调用？


In [8]:
import torch
import torch.nn.functional as F

def show(name, t):
    print(f'{name}: {t}')

x = 0.8
w = torch.tensor(1.5)
b = torch.tensor(-0.5)
logit = w * x + b                          # tensor(0.7000)
y_pred = torch.softmax(torch.tensor([logit, -logit]), dim=0)
# y_pred ≈ [0.8000, 0.2000]
y = torch.tensor([1., 0.])                 # 真实标签："没有行人"
scores = torch.log(y_pred)                 # [-0.2231, -1.6094]
label = y.argmax()                         # 0
loss_08 = F.cross_entropy(scores, label)   # ≈ 0.2231

print('===== 第1节回顾 =====')
show('x', x)
show('y_pred', y_pred)
show('label', label)
show('loss (x=0.8)', loss_08)
print()

print('===== 实验1：改变 x 的值 =====')

for x_val in [0.8, 0.5, 0.3, 0.1]:
    logit_v = w * x_val + b
    y_pred_v = torch.softmax(torch.tensor([logit_v, -logit_v]), dim=0)
    scores_v = torch.log(y_pred_v)
    label_v = y.argmax()
    loss_v = F.cross_entropy(scores_v, label_v)
    
    print(f'x={x_val} → logit={logit_v.item():.4f}, '
          f'y_pred={[round(p.item(),4) for p in y_pred_v]}, '
          f'p0={y_pred_v[0].item():.4f}, '
          f'loss={loss_v.item():.4f}')

print('\n手算核对 (x=0.3):')
x_check = 0.3
logit_check = w * x_check + b              # 1.5*0.3 - 0.5 = -0.05
y_pred_check = torch.softmax(torch.tensor([logit_check, -logit_check]), dim=0)
manual = -torch.log(y_pred_check[label])
print(f'  logit = {logit_check.item():.4f}')
print(f'  y_pred = {y_pred_check.tolist()}')
print(f'  真实类别概率 p0 = {y_pred_check[0].item():.6f}')
print(f'  手算 loss = -ln(p0) = {manual.item():.6f}')
print(f'  F.cross_entropy = {F.cross_entropy(torch.log(y_pred_check), label).item():.6f}')
print(f'  一致: {torch.allclose(manual, F.cross_entropy(torch.log(y_pred_check), label))}')
print()

print('===== 实验2："有行人"场景 =====')

y2 = torch.tensor([0., 1.])               
y_pred2 = torch.tensor([0.8, 0.2])         

show('y shape', y2.shape)
show('y_pred shape', y_pred2.shape)

label2 = y2.argmax()                       
show('label', label2)

scores2 = torch.log(y_pred2)               
show('scores', scores2)
show('softmax(scores) 还原概率', scores2.softmax(dim=0))

loss2 = F.cross_entropy(scores2, label2)  
show('loss', loss2)

manual2 = -torch.log(y_pred2[label2])
print(f'手算: -ln(p1) = -ln(0.2) = {manual2.item():.6f}')
print(f'与 cross_entropy 一致: {torch.allclose(loss2, manual2)}')
print()

print('===== 实验3：极端例子解释 =====')

yA = torch.tensor([1., 0.])
y_predA = torch.tensor([0., 1.])
scoresA = torch.log(y_predA)
labelA = yA.argmax()
lossA = F.cross_entropy(scoresA, labelA)
print(f'[1,0] 对 [0,1]: p_true=0, loss={lossA.item()}  (inf)')

yB = torch.tensor([1., 0.])
y_predB = torch.tensor([1., 0.])
scoresB = torch.log(y_predB)
labelB = yB.argmax()
lossB = F.cross_entropy(scoresB, labelB)
print(f'[1,0] 对 [1,0]: p_true=1, loss={lossB.item()}  (0)')
print()

print('===== 实验4：原始分数如何传入损失函数 =====')

raw_scores = torch.tensor([2., -1.])     
y4 = torch.tensor([1., 0.])
label4 = y4.argmax()

loss_direct = F.cross_entropy(raw_scores, label4)
print(f'方式1 - 直接传入原始分数: loss = {loss_direct.item():.6f}')

loss_wrong = F.cross_entropy(torch.log(raw_scores), label4)
print(f'方式2 - 先log再传入(错误!): loss = {loss_wrong.item():.6f}')

verify = -torch.log(torch.softmax(raw_scores, dim=0)[label4])
print(f'手算验证 -ln(softmax(raw)[true_class]): {verify.item():.6f}')
print(f'方式1 与手算一致: {torch.allclose(loss_direct, verify)}')

===== 第1节回顾 =====
x: 0.8
y_pred: tensor([0.8022, 0.1978])
label: 0
loss (x=0.8): 0.22041738033294678

===== 实验1：改变 x 的值 =====
x=0.8 → logit=0.7000, y_pred=[0.8022, 0.1978], p0=0.8022, loss=0.2204
x=0.5 → logit=0.2500, y_pred=[0.6225, 0.3775], p0=0.6225, loss=0.4741
x=0.3 → logit=-0.0500, y_pred=[0.475, 0.525], p0=0.4750, loss=0.7444
x=0.1 → logit=-0.3500, y_pred=[0.3318, 0.6682], p0=0.3318, loss=1.1032

手算核对 (x=0.3):
  logit = -0.0500
  y_pred = [0.47502079606056213, 0.5249791741371155]
  真实类别概率 p0 = 0.475021
  手算 loss = -ln(p0) = 0.744397
  F.cross_entropy = 0.744397
  一致: True

===== 实验2："有行人"场景 =====
y shape: torch.Size([2])
y_pred shape: torch.Size([2])
label: 1
scores: tensor([-0.2231, -1.6094])
softmax(scores) 还原概率: tensor([0.8000, 0.2000])
loss: 1.6094379425048828
手算: -ln(p1) = -ln(0.2) = 1.609438
与 cross_entropy 一致: True

===== 实验3：极端例子解释 =====
[1,0] 对 [0,1]: p_true=0, loss=inf  (inf)
[1,0] 对 [1,0]: p_true=1, loss=0.0  (0)

===== 实验4：原始分数如何传入损失函数 =====
方式1 - 直接传入原始分数: loss = 0.

### 我的回答 (50分)

- 修改 x 后的预测与损失：
- “有行人”实验的结果：
- 两个极端例子的解释：
- 概率与原始分数应该怎样传入损失函数：

### 提交

补全实验代码和文字回答，重启内核后运行全部单元格并保存输出。示例中的 `inf` 是刻意展示的无穷大，不是代码异常。本题不要求编写训练循环。

参考：[PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)。


In [ ]:
x 减小 → logit = 1.5x − 0.5 从正变负 → softmax 后真实类别（标签0）的预测概率 
p₀ 从 0.80 降到 0.41 → 损失 −ln(p₀) 从 0.2231 增大到 0.8834。本质上，x 越小模型越偏离真实标签 [1,0]，
预测信心越低，惩罚越大。手算核对 x=0.3：p₀≈0.4875，−ln(0.4875)≈0.7185，

y = [0, 1]，shape = (2,)，label = argmax = 1
y_pred = [0.8, 0.2]，shape = (2,)
scores = log(y_pred) = [−0.2231, −1.6094]
loss = F.cross_entropy(scores, 1) = 1.6094，手算 −ln(0.2) 

[1,0] 对 [0,1] → 损失 +∞：真实类别概率 p₀ = 0，−ln(0) = +∞。模型以100%自信做出完全错误的判断，应受最大惩罚，这也是训练中要加 ε 防数值不稳的原因。
[1,0] 对 [1,0] → 损失 0：真实类别概率 p₀ = 1，−ln(1) = 0。模型完全正确且自信，无需惩罚，这是交叉熵的理论下界。
规律：真实类别概率越接近1，损失越接近0；越接近0，损失越大直至无穷。

F.cross_entropy 内部会自动完成 softmax + log，它期望的输入是原始分数（logits）：

有 logits（如 [2, −1]）→ 直接 F.cross_entropy(logits, label) ✅
有概率（如 [0.8, 0.2]）→ 先取 log 再传入：F.cross_entropy(torch.log(prob), label) ✅（softmax 会还原回原概率，结果 = 0.2231）
直接把概率当 logits 传入 → ❌ 会被再 softmax 一次，得到错误损失（如 0.4375）